# 01 Results Overview
## 결과 파일 개요

원패널을 다시 적재하지 않고, 포트폴리오에 포함할 수 있는 결과 CSV의 구조를 확인한다.
Colab 실행 로그는 제외하고 최종 결과표가 어떤 분석 축을 담고 있는지만 정리한다.


## Setup
### 설정

결과표 위치와 공통 plotting 설정을 불러온다.


In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "model_results"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

CATEGORY_LABELS = {
    "cereal": "Cereal",
    "canned_soup": "Canned soup",
    "bottled_juices": "Bottled juices",
    "cookies": "Cookies",
}


In [2]:
def read_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def category_from_path(path):
    name = Path(path).name
    for category in CATEGORY_LABELS:
        if name.startswith(category):
            return category
    return None


def load_category_files(suffix):
    frames = []
    for path in sorted(TABLE_DIR.glob(f"*_only/*_{suffix}.csv")):
        frame = read_csv(path)
        if "category" not in frame.columns:
            frame["category"] = category_from_path(path)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f"No files matched suffix: {suffix}")
    return pd.concat(frames, ignore_index=True)


def add_category_label(frame):
    out = frame.copy()
    out["category_label"] = out["category"].map(CATEGORY_LABELS).fillna(out["category"])
    return out


def save_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path


## Result inventory
### 결과 파일 목록

zip에서 추출한 CSV 파일을 분석 영역별로 집계한다.


In [3]:
all_csv_files = sorted(TABLE_DIR.rglob("*.csv"))
inventory = pd.DataFrame({
    "path": [str(path.relative_to(TABLE_DIR)) for path in all_csv_files],
    "file_name": [path.name for path in all_csv_files],
    "analysis_group": [path.relative_to(TABLE_DIR).parts[0] for path in all_csv_files],
})
summary = inventory.groupby("analysis_group", as_index=False).size().rename(columns={"size": "n_files"})
display(summary)
print(f"Total CSV files: {len(inventory)}")


,analysis_group,n_files
0,bottled_juices_only,10
1,canned_soup_only,10
2,cereal_2way_fe_iv_decomposition_results.csv,1
3,cereal_only,10
4,cookies_only,10
5,dynamic_checks,9
6,robustness,4


Total CSV files: 54


결과 파일은 카테고리별 FE-IV 결과와 dynamic robustness 결과로 나뉜다.
포트폴리오 본문에서는 실행 로그가 아니라 최종 결과표만 사용한다.


## Category coverage
### 카테고리 범위

카테고리별 combined result 파일을 읽어 분석 대상의 규모를 확인한다.


In [4]:
combined = load_category_files("combined_model_results")
combined = add_category_label(combined)
coverage = (
    combined.groupby("category_label", as_index=False)
    .agg(
        n_result_rows=("method", "size"),
        max_observations=("n_obs", "max"),
        max_store_upc_clusters=("n_store_upc_clusters", "max"),
        max_weeks=("n_weeks", "max"),
    )
    .sort_values("max_observations", ascending=False)
)
display(coverage)


,category_label,n_result_rows,max_observations,max_store_upc_clusters,max_weeks
3,Cookies,36,13150237,78134.0,388.0
1,Canned soup,36,6862202,32693.0,378.0
2,Cereal,36,6451609,36438.0,366.0
0,Bottled juices,36,6113366,36386.0,392.0


네 개 카테고리 모두 수백만 건 단위의 store-upc-week 관측치를 사용했다.
GitHub에는 원패널 대신 결과 CSV를 포함해, 분석 결론과 그림을 재생성할 수 있게 둔다.


## Model families
### 모델 계열

결과표에 포함된 추정 방식과 주요 산출물을 확인한다.


In [5]:
model_family = (
    combined.groupby(["method", "outcome_name"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["method", "outcome_name"])
)
display(model_family)


,method,outcome_name,n_rows
0,FD-PLIV-DML,own_sales,16
1,FE-IV,category_total_sales,20
2,FE-IV,own_pseudo_logit,20
3,FE-IV,own_sales,28
4,FE-IV,rival_mfr_sales,20
5,FE-IV,same_com_code_sales,20
6,FE-IV,same_mfr_sales,20
